In [3]:
import sys
from pathlib import Path

# Tìm thư mục gốc dự án
root = Path.cwd()

if root.name == "train":
    root = root.parent

sys.path.insert(0, str(root))

import torch

from src.data.dataset import create_dataloaders
from src.models.lstm import LSTMLanguageModel

In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

train_loader, valid_loader, test_loader, vocab_size = (
    create_dataloaders()
)

model = LSTMLanguageModel(vocab_size=vocab_size, embedding_dim=256, hidden_dim=256, num_layers=2, dropout=0.2 ).to(device)

print("Device:", device)
print("Vocabulary size:", vocab_size)

print(
    "Model parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}"
)
print(model)

Device: cuda
Vocabulary size: 10000
Model parameters: 6,182,672
LSTMLanguageModel(
  (embedding): Embedding(10000, 256)
  (lstm): LSTM(256, 256, num_layers=2, batch_first=True, dropout=0.2)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=256, out_features=10000, bias=True)
)


Kiểm tra Forward Pass

In [3]:
X, Y = next(iter(train_loader))

X = X.to(device)

model.eval()

with torch.no_grad():
    logits = model(X)

print("Input shape:", X.shape)
print("Target shape:", Y.shape)
print("Output shape:", logits.shape)

assert logits.shape == (
    X.shape[0],
    X.shape[1],
    vocab_size
)

print("LSTM hoạt động thành công!")

Input shape: torch.Size([32, 64])
Target shape: torch.Size([32, 64])
Output shape: torch.Size([32, 64, 10000])
LSTM hoạt động thành công!


Khai báo Loss và Optimizer

In [4]:
import torch
import torch.nn as nn
import math
import time
import pandas as pd

from tqdm.auto import tqdm

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

Viết hàm huấn luyện một Epoch

In [5]:
def train_one_epoch(model, train_loader, optimizer, criterion):

    model.train()

    total_loss = 0
    total_tokens = 0

    progress = tqdm(
        train_loader,
        desc="Training"
    )

    for X, Y in progress:

        X = X.to(device)
        Y = Y.to(device)

        # Xóa gradient cũ
        optimizer.zero_grad()

        # Forward
        logits = model(X)

        # Tính loss
        loss = criterion(
            logits.reshape(-1, vocab_size),
            Y.reshape(-1)
        )

        # Backpropagation
        loss.backward()

        # Hạn chế exploding gradient
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        # Cập nhật trọng số
        optimizer.step()

        # Thống kê
        num_tokens = Y.numel()

        total_loss += loss.item() * num_tokens
        total_tokens += num_tokens

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    avg_loss = total_loss / total_tokens

    ppl = math.exp(avg_loss)

    return avg_loss, ppl

Viết hàm Validation

In [6]:
@torch.no_grad()
def evaluate(model, data_loader, criterion):

    model.eval()

    total_loss = 0
    total_tokens = 0

    for X, Y in tqdm(data_loader, desc="Evaluating"):

        X = X.to(device)
        Y = Y.to(device)

        logits = model(X)

        loss = criterion(
            logits.reshape(-1, vocab_size),
            Y.reshape(-1)
        )

        num_tokens = Y.numel()

        total_loss += loss.item() * num_tokens
        total_tokens += num_tokens

    avg_loss = total_loss / total_tokens

    ppl = math.exp(avg_loss)

    return avg_loss, ppl

Huấn luyện chính thức và lưu mô hình tốt nhất

In [7]:
from src.models.lstm import LSTMLanguageModel

model = LSTMLanguageModel(vocab_size=vocab_size, embedding_dim=256, hidden_dim=256, num_layers=2, dropout=0.2 ).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
from pathlib import Path
import time
import torch
import pandas as pd

# ==================================
# 1. CẤU HÌNH THÍ NGHIỆM
# ==================================

CONTEXT_LENGTH = 128
BATCH_SIZE = 16

EPOCHS = 50
PATIENCE = 5

EXPERIMENT_NAME = f"word_c{CONTEXT_LENGTH}"

# ==================================
# 2. TẠO DATALOADER
# ==================================

train_loader, valid_loader, test_loader, vocab_size = (
    create_dataloaders(
        context_length=CONTEXT_LENGTH,
        batch_size=BATCH_SIZE
    )
)

# ==================================
# 3. KHỞI TẠO LẠI MODEL
# ==================================

model = LSTMLanguageModel(
    vocab_size=vocab_size,
    embedding_dim=256,
    hidden_dim=256,
    num_layers=2,
    dropout=0.2
).to(device)

criterion = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# ==================================
# 4. TẠO THƯ MỤC LƯU KẾT QUẢ
# ==================================

CHECKPOINT_DIR = (
    root / "checkpoints" / "lstm" / EXPERIMENT_NAME
)

RESULTS_DIR = (
    root / "results" / "lstm" / EXPERIMENT_NAME
)

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ==================================
# 5. HUẤN LUYỆN
# ==================================

best_valid_loss = float("inf")
patience_counter = 0

history = []

start_time = time.time()

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    train_loss, train_ppl = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion
    )

    valid_loss, valid_ppl = evaluate(
        model,
        valid_loader,
        criterion
    )

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train PPL: {train_ppl:.2f}")
    print(f"Valid Loss: {valid_loss:.4f}")
    print(f"Valid PPL: {valid_ppl:.2f}")

    history.append({
        "epoch": epoch + 1,
        "context_length": CONTEXT_LENGTH,
        "batch_size": BATCH_SIZE,
        "train_loss": train_loss,
        "train_ppl": train_ppl,
        "valid_loss": valid_loss,
        "valid_ppl": valid_ppl
    })

    # Lưu best model
    if valid_loss < best_valid_loss:

        best_valid_loss = valid_loss
        patience_counter = 0

        torch.save(
            model.state_dict(),
            CHECKPOINT_DIR / "best_model.pt"
        )

        print("Đã lưu best model!")

    else:
        patience_counter += 1

        print(
            f"Không cải thiện: "
            f"{patience_counter}/{PATIENCE}"
        )

    # Lưu kết quả từng epoch
    pd.DataFrame(history).to_csv(
        RESULTS_DIR / "lstm_results.csv",
        index=False
    )

    # Early Stopping
    if patience_counter >= PATIENCE:

        print("Early Stopping!")
        break

# ==================================
# 6. TỔNG THỜI GIAN
# ==================================

if device.type == "cuda":
    torch.cuda.synchronize()

training_time = time.time() - start_time

print(
    f"\nTraining time: "
    f"{training_time / 60:.2f} phút"
)

# ==================================
# 7. ĐÁNH GIÁ TEST SET
# ==================================

model.load_state_dict(
    torch.load(
        CHECKPOINT_DIR / "best_model.pt",
        map_location=device,
        weights_only=True
    )
)

test_loss, test_ppl = evaluate(
    model,
    test_loader,
    criterion
)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test PPL: {test_ppl:.2f}")

# ==================================
# 8. LƯU BẢNG TỔNG HỢP
# ==================================

summary = pd.DataFrame([{
    "model": "LSTM",
    "tokenizer": "word",
    "context_length": CONTEXT_LENGTH,
    "batch_size": BATCH_SIZE,
    "epochs_trained": len(history),
    "parameters": sum(
        p.numel() for p in model.parameters()
    ),
    "best_valid_loss": best_valid_loss,
    "best_valid_ppl": float(__import__("math").exp(best_valid_loss)),
    "test_loss": test_loss,
    "test_ppl": test_ppl,
    "training_time_seconds": training_time
}])

summary.to_csv(
    RESULTS_DIR / "summary.csv",
    index=False
)

display(summary)


Epoch 1/50


Training:   0%|          | 0/1042 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/109 [00:00<?, ?it/s]

Train Loss: 6.0067
Train PPL: 406.15
Valid Loss: 5.3231
Valid PPL: 205.01
Đã lưu best model!

Epoch 2/50


Training:   0%|          | 0/1042 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
print("Train samples:", len(train_loader.dataset))
print("Train batches:", len(train_loader))
print("Batch size:", train_loader.batch_size)
print("Tokens per sample:", train_loader.dataset.context_length)
print("Total train tokens:", len(train_loader.dataset.tokens))

Train samples: 33335
Train batches: 1042
Batch size: 32
Tokens per sample: 64
Total train tokens: 2133462
